In [11]:
%pip install yfinance tensorflow

import yfinance as yf
import pandas as pd
import numpy as np
import random
import tensorflow as tf

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, matthews_corrcoef
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [12]:
tickers = ["QQQ", "BB", "M", "CMG"]

SENTIMENT_FILES = {
    "QQQ": "../Sentiment/QQQ_Sentiment.csv",
    "BB": "../Sentiment/BB_Sentiment.csv",
    "M": "../Sentiment/M_Sentiment.csv",
    "CMG": "../Sentiment/CMG_Sentiment.csv"
}

In [ ]:
def merge_sentiment(price, sentiment_path):
    sentiment_df = pd.read_csv(sentiment_path)

    sentiment_df["date"] = pd.to_datetime(sentiment_df["date"], errors="coerce")
    sentiment_df["sentiment"] = pd.to_numeric(sentiment_df["sentiment"], errors="coerce")
    sentiment_df = sentiment_df.dropna(subset=["date", "sentiment"])

    if getattr(sentiment_df["date"].dt, "tz", None) is not None:
        sentiment_df["date"] = sentiment_df["date"].dt.tz_localize(None)

    sentiment_df["date"] = sentiment_df["date"].dt.normalize()

    sentiment_df = sentiment_df.groupby("date", as_index=False)["sentiment"].mean()

    price = price.copy()
    price.index = pd.to_datetime(price.index)

    if getattr(price.index, "tz", None) is not None:
        price.index = price.index.tz_localize(None)

    price.index = price.index.normalize()

    price_reset = price.reset_index()
    price_reset = price_reset.rename(columns={price_reset.columns[0]: "date"})

    price_reset["date"] = pd.to_datetime(price_reset["date"], errors="coerce").dt.normalize()

    merged = price_reset.merge(sentiment_df, on="date", how="left")

    merged["sentiment"] = merged["sentiment"].fillna(0)

    merged = merged.set_index("date")

    return merged

In [14]:
prices = {}

for ticker_symbol in tickers:
    ticker = yf.Ticker(ticker_symbol)
    price = ticker.history(start="2009-02-14", end="2020-06-11")

    price = merge_sentiment(price, SENTIMENT_FILES[ticker_symbol])

    prices[ticker_symbol] = price

    print(f"\n{ticker_symbol}:")
    print(price[["Open", "Close", "sentiment"]].head())


QQQ:
                 Open      Close  sentiment
date                                       
2009-02-17  25.455397  25.222336        0.0
2009-02-18  25.386343  25.239601        0.0
2009-02-19  25.394973  24.851166        0.0
2009-02-20  24.574945  24.920221        0.0
2009-02-23  25.058332  24.048403        0.0

BB:
                 Open      Close  sentiment
date                                       
2009-02-17  46.540001  44.639999        0.0
2009-02-18  45.500000  42.099998        0.0
2009-02-19  42.419998  42.090000        0.0
2009-02-20  40.610001  39.150002        0.0
2009-02-23  39.840000  37.419998        0.0

M:
                Open     Close  sentiment
date                                     
2009-02-17  4.532084  4.543329        0.0
2009-02-18  4.593936  4.436494        0.0
2009-02-19  4.560199  4.307167        0.0
2009-02-20  4.245314  4.419625        0.0
2009-02-23  4.498346  4.160970        0.0

CMG:
              Open   Close  sentiment
date                           

In [ ]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    price["Score"] = (price["Close"].shift(-1) > price["Close"]).astype(int)

    price["Change"]      = price["Close"].shift(1) - price["Open"].shift(1)
    price["%Change"]     = (price["Close"].shift(1) - price["Open"].shift(1)) / price["Open"].shift(1)
    price["CloseToOpen"] = price["Open"].shift(1) - price["Close"].shift(1)
    price["YestScore"]   = price["Score"].shift(1)
    price["5DayMean"]    = price["Close"].rolling(5).mean().shift(1)
    price["5DayVoli"]    = price["Close"].rolling(5).std().shift(1)
    price["5DayPerf"]    = price["Score"].rolling(5).sum().shift(1)

    price.dropna(inplace=True)
    prices[ticker_symbol] = price

In [16]:
features = [
    "Change", "%Change", "CloseToOpen", "YestScore",
    "5DayMean", "5DayVoli", "5DayPerf", "sentiment"
]

window_size = 5
all_results = {}

def make_windows(feature_data, target_data, dates, window_size):
    X = []
    y = []
    d = []

    for i in range(window_size, len(feature_data)):
        X.append(feature_data[i-window_size:i])
        y.append(target_data[i])
        d.append(dates[i])

    return np.array(X), np.array(y), np.array(d)

In [17]:
for ticker_symbol in tickers:
    price = prices[ticker_symbol]

    priceTrain = price[price.index <= "2018-06-11"].copy()
    priceTest  = price[price.index > "2018-06-11"].copy()

    scaler = StandardScaler()

    xTrain_raw = priceTrain[features]
    xTest_raw  = priceTest[features]

    xTrain_scaled = scaler.fit_transform(xTrain_raw)
    xTest_scaled  = scaler.transform(xTest_raw)

    yTrain = priceTrain["Score"].values
    yTest  = priceTest["Score"].values

    train_dates = priceTrain.index
    test_dates  = priceTest.index

    X_train, y_train, d_train = make_windows(xTrain_scaled, yTrain, train_dates, window_size)

    combined_X = np.vstack([xTrain_scaled[-window_size:], xTest_scaled])
    combined_y = np.concatenate([yTrain[-window_size:], yTest])
    combined_d = np.concatenate([train_dates[-window_size:], test_dates])

    X_test, y_test, d_test = make_windows(combined_X, combined_y, combined_d, window_size)

    model = Sequential([
        LSTM(32, input_shape=(window_size, len(features))),
        Dropout(0.2),
        Dense(1, activation="sigmoid")
    ])

    model.compile(
        optimizer="adam",
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )

    early_stop = EarlyStopping(
        monitor="val_loss",
        patience=5,
        restore_best_weights=True
    )

    model.fit(
        X_train, y_train,
        validation_split=0.2,
        epochs=20,
        batch_size=32,
        callbacks=[early_stop],
        verbose=0
    )

    probs = model.predict(X_test, verbose=0).flatten()
    preds = (probs >= 0.5).astype(int)

    results = pd.DataFrame({
        "Date": d_test,
        "Score": y_test,
        "Prediction": preds,
        "Probability": probs
    })

    all_results[ticker_symbol] = results

    print(f"\nFinished {ticker_symbol}")
    print(results.head())

C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished QQQ
        Date  Score  Prediction  Probability
0 2018-06-12      0           1     0.553024
1 2018-06-13      1           1     0.556257
2 2018-06-14      0           1     0.511609
3 2018-06-15      0           1     0.557127
4 2018-06-18      0           0     0.476044


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished BB
        Date  Score  Prediction  Probability
0 2018-06-12      0           0     0.442054
1 2018-06-13      0           0     0.453297
2 2018-06-14      1           0     0.471152
3 2018-06-15      0           0     0.481562
4 2018-06-18      0           0     0.478746


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished M
        Date  Score  Prediction  Probability
0 2018-06-12      0           1     0.533560
1 2018-06-13      0           1     0.505766
2 2018-06-14      1           1     0.571624
3 2018-06-15      1           1     0.581151
4 2018-06-18      1           1     0.603818


C:\Users\ericf\AppData\Roaming\Python\Python312\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)



Finished CMG
        Date  Score  Prediction  Probability
0 2018-06-12      0           0     0.482682
1 2018-06-13      1           1     0.525776
2 2018-06-14      1           1     0.532483
3 2018-06-15      1           0     0.487559
4 2018-06-18      1           0     0.464553


In [18]:
for ticker_symbol, results in all_results.items():
    score = results["Score"]
    pred = results["Prediction"]

    accuracy  = accuracy_score(score, pred)
    precision = precision_score(score, pred, zero_division=0)
    recall    = recall_score(score, pred, zero_division=0)
    f1        = f1_score(score, pred, zero_division=0)
    mcc       = matthews_corrcoef(score, pred)

    print(f"\n{ticker_symbol}:")
    print("Mean:     ", results["Score"].mean())
    print("Accuracy: ", accuracy)
    print("Precision:", precision)
    print("Recall:   ", recall)
    print("F1:       ", f1)
    print("MCC:      ", mcc)


QQQ:
Mean:      0.5765407554671969
Accuracy:  0.5149105367793241
Precision: 0.5761589403973509
Recall:    0.6
F1:        0.5878378378378378
MCC:       -0.0009471912832606722

BB:
Mean:      0.4691848906560636
Accuracy:  0.5208747514910537
Precision: 0.4489795918367347
Recall:    0.09322033898305085
F1:        0.1543859649122807
MCC:       -0.013301228191874546

M:
Mean:      0.48111332007952284
Accuracy:  0.48906560636182905
Precision: 0.4740484429065744
Recall:    0.5661157024793388
F1:        0.5160075329566854
MCC:       -0.0164318589716122

CMG:
Mean:      0.5546719681908548
Accuracy:  0.47713717693836977
Precision: 0.5465116279069767
Recall:    0.33691756272401435
F1:        0.41685144124168516
MCC:       -0.011835875000710873


In [19]:
def compute_sharpe_ratio(daily_returns, annualize=True, periods_per_year=252):
    daily_returns = np.asarray(daily_returns, dtype=float)
    daily_returns = daily_returns[~np.isnan(daily_returns)]

    if len(daily_returns) < 2:
        return np.nan

    std_ret = np.std(daily_returns, ddof=1)
    if std_ret == 0:
        return np.nan

    sharpe = np.mean(daily_returns) / std_ret

    if annualize:
        sharpe *= np.sqrt(periods_per_year)

    return sharpe

In [20]:
for ticker_symbol, results in all_results.items():
    totalIn = 0
    totalOut = 0
    totalInvested = 0
    totalStoodOut = 0
    alltradein = 0
    alltradeout = 0
    perftradein = 0
    perftradeout = 0

    price = prices[ticker_symbol]

    model_daily_returns = []
    buyhold_daily_returns = []
    perfect_daily_returns = []

    for date, pred, score in zip(results["Date"], results["Prediction"], results["Score"]):
        current_close = price.loc[date, "Close"]

        next_idx = price.index.get_loc(date) + 1
        if next_idx >= len(price):
            continue

        next_close = price.iloc[next_idx]["Close"]

        ret_decimal = (next_close - current_close) / current_close
        ret_percent = ret_decimal * 100

        if pred == 1:
            totalIn += 100
            totalInvested += 1
            totalOut += 100 + ret_percent
            model_daily_returns.append(ret_decimal)
        else:
            totalStoodOut += 1

        alltradein += 100
        alltradeout += 100 + ret_percent
        buyhold_daily_returns.append(ret_decimal)

        if score == 1:
            perftradein += 100
            perftradeout += 100 + ret_percent
            perfect_daily_returns.append(ret_decimal)

    model_sharpe = compute_sharpe_ratio(model_daily_returns)
    buyhold_sharpe = compute_sharpe_ratio(buyhold_daily_returns)
    perfect_sharpe = compute_sharpe_ratio(perfect_daily_returns)

    print(f"\n{ticker_symbol}:")
    print(
        f"  Model:    Invested ${totalIn:.0f}, Returned ${totalOut:.2f}, "
        f"Profit ${totalOut - totalIn:.2f} ({totalInvested} trades, {totalStoodOut} skipped), "
        f"Sharpe {model_sharpe:.3f}"
    )
    print(
        f"  Buy&Hold: Invested ${alltradein:.0f}, Returned ${alltradeout:.2f}, "
        f"Profit ${alltradeout - alltradein:.2f}, Sharpe {buyhold_sharpe:.3f}"
    )
    print(
        f"  Perfect:  Invested ${perftradein:.0f}, Returned ${perftradeout:.2f}, "
        f"Profit ${perftradeout - perftradein:.2f}, Sharpe {perfect_sharpe:.3f}"
    )


QQQ:
  Model:    Invested $30200, Returned $30231.16, Profit $31.16 (302 trades, 200 skipped), Sharpe 1.083
  Buy&Hold: Invested $50200, Returned $50243.01, Profit $43.01, Sharpe 0.783
  Perfect:  Invested $29000, Returned $29296.20, Profit $296.20, Sharpe 13.111

BB:
  Model:    Invested $4900, Returned $4857.78, Profit $-42.22 (49 trades, 453 skipped), Sharpe -2.102
  Buy&Hold: Invested $50200, Returned $50150.43, Profit $-49.57, Sharpe -0.462
  Perfect:  Invested $23600, Returned $24110.42, Profit $510.42, Sharpe 14.504

M:
  Model:    Invested $28800, Returned $28703.21, Profit $-96.79 (288 trades, 214 skipped), Sharpe -1.308
  Buy&Hold: Invested $50200, Returned $50095.15, Profit $-104.85, Sharpe -0.821
  Perfect:  Invested $24200, Returned $24783.08, Profit $583.08, Sharpe 12.471

CMG:
  Model:    Invested $17200, Returned $17268.03, Profit $68.03 (172 trades, 330 skipped), Sharpe 1.937
  Buy&Hold: Invested $50200, Returned $50295.25, Profit $95.25, Sharpe 1.186
  Perfect:  Inve